### =============================================================================
### HOUSE PRICE PREDICTION - EXPLORATORY DATA ANALYSIS
### =============================================================================
### Author: Beksultan a.k.a rsuvbe
### Date: 23.03.2026
### Description: Initial data exploration and quality assessment
### =============================================================================


In [ ]:


# # 1. Import Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns



# Plot settings
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')


# # 2. Load Data

train_df = pd.read_csv('../data/train.csv')

print(f"Dataset Shape: {train_df.shape[0]} rows × {train_df.shape[1]} columns")
print(f"Memory Usage: {train_df.memory_usage(deep=True).sum() / 1024 ** 2:.2f} MB")

# # 3. Data Overview

print("\n" + "="*60)
print("FIRST 5 ROWS")
print("="*60)
display(train_df.head())

print("\n" + "="*60)
print("DATA TYPES SUMMARY")
print("="*60)
dtype_summary = train_df.dtypes.value_counts()
print(dtype_summary)

# Column names by type

numeric_cols = train_df.select_dtypes(include = [np.number]).columns.tolist()
categorical_cols = train_df.select_dtypes(include = ['object', 'str']).columns.tolist()

print(f"\nNumeric columns: {len(numeric_cols)}")
print(f"Categorical columns: {len(categorical_cols)}")

## Target Variable Analysis

#Statistical Summary 

print("\n" + "="* 60)
print("TARGET VARIABLE (SalePrice) - STATISTICS")
print("="*60)
print(train_df['SalePrice'].describe())

#Distribution plot

fig, axes = plt.subplots(1,2, figsize=(14,5))

#Histogram

axes[0].hist(train_df['SalePrice'], bins = 50, edgecolor='black', alpha = 0.7)
axes[0].set_xlabel('SalePrice ($)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Sale Price')
axes[0].grid(True, alpha=0.3)

# Boxplot (detect outliers)

axes[1].boxplot(train_df['SalePrice'], vert=False)
axes[1].set_xlabel('Sale Price ($)')
axes[1].set_title('BoxPlot - Detect Outliers')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/01_target_distribution.png', dpi = 150)
plt.show()

# Check for skewness 
skewness = train_df['SalePrice'].skew()
kurtosis = train_df['SalePrice'].kurtosis()
print(f"\nSkewness: {skewness:.2f}")
print(f"Kurtosis: {kurtosis:.2f}")
print("-> Positive skewness indicates right-tailed distribution")
print("-> May need log transformation for modeling")

## 5. Missing Values Analysis

missing_df = pd.DataFrame({
	'Missing_Count': train_df.isnull().sum(),
	'Missing_Percent': (train_df.isnull().sum() / len(train_df)) * 100
})

missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Percent', ascending = False)

print('='*60)
print("MISING VALUES ANALYSIS")
print("="*60)

if len(missing_df) > 0:
	print(f"Columns with missing values: {len(missing_df)} out of {train_df.shape[1]}")
	print("\nTop 15 columns with missing values: ")
	print(missing_df.head(15))
else:
	print("No missing values found!")

#Visualize missing values

if len(missing_df) > 0:
	plt.figure(figsize=(10,8))
	top_missing = missing_df.head(15)
	plt.barh(top_missing.index, top_missing['Missing_Percent'])
	plt.xlabel('Missing Percentage (%)')
	plt.title('Top 15 Columns with Missing Values')
	plt.gca().invert_yaxis()
	plt.grid(True, alpha=0.3)
	plt.tight_layout()
	plt.savefig('../results/02_missing_values.png', dpi=150)
	plt.show()



# # 6. Numeric Features Correlation with Target

#Correlation matrix (numeric features only)
numeric_df = train_df[numeric_cols]
correlation_matrix = numeric_df.corr()

#Correlation with Sale Price
price_correlation = correlation_matrix['SalePrice'].sort_values(ascending=False)

print("\n" + "="*60)
print("TOP 15 FEATURES CORRELATED WITH SALE PRICE")
print("="*60)
print(price_correlation.head(16)) #Include Sale Price itself

#Heatmap of top correlations 
top_features= price_correlation.head(11).index.tolist()
plt.figure(figsize=(12,10))
sns.heatmap(train_df[top_features].corr(), 
						annot=True,
						fmt='.2f',
						cmap='coolwarm',
						center=0,
						square = True,
						linewidth = 1)
plt.title('Correlation Heatmap - Top Features with Sale Price')
plt.tight_layout()
plt.savefig('../results/03_correlation_heatmap.png', dpi = 150)
plt.show()


# # 7. Key Feature Distributions

# Plot distributions of top correlated features
top_numeric_features = ['OverallQual', 'GrLivArea', 'GarageCars', 'GarageArea', 'TotalBsmtSF']

fig, axes = plt.subplots(2,3, figsize = (15,10))
axes = axes.flatten()

for idx, feature in enumerate(top_numeric_features):
	if feature in train_df.columns:
		axes[idx].scatter(train_df[feature], train_df['SalePrice'], alpha=0.5, s=20)
		axes[idx].set_xlabel(feature)
		axes[idx].set_ylabel('Sale Price ($)')
		axes[idx].set_title(f'{feature} vs Sale Price')
		axes[idx].grid(True, alpha=0.3)

#Hide Empty Subplot
axes[-1].axis('off')

plt.tight_layout()
plt.savefig('../results/04_feature_scatterplots.png', dpi=150)
plt.show()

## 8. Categorical Features Analysis

# Top categorical features by cardinality

categorical_cardinality = train_df[categorical_cols].nunique().sort_values(ascending=False)

print("\n" + "="*60)
print("CATEGORICAL FEATURES - CARDINALITY")
print("="*60)
print(categorical_cardinality.head(15))

#Example: Neighborhood vs Sale Price

if 'Neighborhood' in train_df.columns:
	plt.figure(figsize=(14,6))
	neighborhood_price = train_df.groupby('Neighborhood')['SalePrice'].median().sort_values(ascending=False)
	neighborhood_price.plot(kind='bar')
	plt.xlabel('Neighborhood')
	plt.ylabel('Median Sale Price')
	plt.title('Median Sale Price by Neighborhood')
	plt.xticks(rotation=45, ha='right')
	plt.grid(True,alpha=0.3, axis='y')
	plt.tight_layout()
	plt.savefig('../results/05_neighborhood_price.png', dpi= 150)
	plt.show()

# # 9. Summary & Next Steps

print("\n" + "="*60)
print("EDA SUMMARY & RECOMMENDATIONS")
print("="*60)

print(""" 
			KEY FINDINGS: 
			1. DATASET: {rows} row x {columns} columns
			2. TARGET VARIABLE (SalePrice): 
				- MEAN: ${mean_price:,.0f}
				-	MEDIAN: ${median_price:,.0f}
				- SKEWNESS: {skew:.2f} (right-skew -> consider log transformation)
			3. MISSING VALUES: {missing_cols} columns have missing data
				- HIGHEST: {top_missing}
			4. TOP CORRELATED FEATURES:
				- OverallQual, GrLivArea, GarageCars, TotalBsmtSF
			5. CATEGORICAL FEATURES: {cat_cols} columns need encoding
			
			NEXT STEPS:
			1. Handle missing values (imputation or removal)
			2. Encode categorical features (One-Hot or Label Encoding)
			3. Consider log transformations of SalePrice
			4. Remove or cap outliers (GrLivArea > 4000)
			5. Split data into train/validation sets
			6. Train baseline Linear Regression Model
			""".format(rows = train_df.shape[0],
								 columns = train_df.shape[1],
								 mean_price = train_df['SalePrice'].mean(),
								 median_price = train_df['SalePrice'].median(),
								 skew = train_df['SalePrice'].skew(),
								 missing_cols = len(missing_df),
								 top_missing = missing_df.index[0] if len(missing_df) > 0 else 'None',
								 cat_cols = len(categorical_cols)
			))

print("\nEXPLORATORY DATA ANALYSIS COMPLETE")

